In [1]:
# ============================================================
# YUVA INTERN - WEEK 4
# AI PROJECT DEPLOYMENT & CAPSTONE
# Project: Iris Flower Prediction using Flask API
# ============================================================

# ------------------------------------------------------------
# 1. INSTALL REQUIRED LIBRARIES
# ------------------------------------------------------------

!pip install flask joblib scikit-learn requests -q


# ------------------------------------------------------------
# 2. IMPORT LIBRARIES
# ------------------------------------------------------------

import numpy as np
import joblib
import os

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


# ------------------------------------------------------------
# 3. LOAD DATASET
# ------------------------------------------------------------

iris = load_iris()

X = iris.data
y = iris.target

print("Dataset loaded successfully!")
print("Dataset shape:", X.shape)
print("Target names:", iris.target_names)


# ------------------------------------------------------------
# 4. SPLIT DATA INTO TRAINING AND TESTING
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nData split completed!")
print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


# ------------------------------------------------------------
# 5. DATA PREPROCESSING
# ------------------------------------------------------------

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nData preprocessing completed!")


# ------------------------------------------------------------
# 6. CREATE AND TRAIN MACHINE LEARNING MODEL
# ------------------------------------------------------------

model = LogisticRegression()

model.fit(X_train_scaled, y_train)

print("\nModel training completed!")


# ------------------------------------------------------------
# 7. TEST THE MODEL
# ------------------------------------------------------------

y_pred = model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)

print("\n================ MODEL EVALUATION ================")
print("Accuracy:", accuracy)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=iris.target_names
    )
)


# ------------------------------------------------------------
# 8. SAVE THE TRAINED MODEL USING JOBLIB
# ------------------------------------------------------------

joblib.dump(model, "iris_model.pkl")
joblib.dump(scaler, "scaler.pkl")

print("\nModel saved as: iris_model.pkl")
print("Scaler saved as: scaler.pkl")


# ------------------------------------------------------------
# 9. CHECK SAVED FILES
# ------------------------------------------------------------

print("\nSaved files:")

for file in os.listdir():
    if file.endswith(".pkl"):
        print("-", file)


# ------------------------------------------------------------
# 10. CREATE FLASK API
# ------------------------------------------------------------

flask_code = '''

from flask import Flask, request, jsonify
import joblib
import numpy as np

app = Flask(__name__)

# Load trained model and scaler
model = joblib.load("iris_model.pkl")
scaler = joblib.load("scaler.pkl")

# Class names
class_names = [
    "setosa",
    "versicolor",
    "virginica"
]


# --------------------------------------------------------
# HOME ROUTE
# --------------------------------------------------------

@app.route("/")
def home():

    return jsonify({
        "message": "Iris Flower Prediction API is running!",
        "endpoint": "/predict",
        "method": "POST"
    })


# --------------------------------------------------------
# PREDICTION ROUTE
# --------------------------------------------------------

@app.route("/predict", methods=["POST"])
def predict():

    try:

        # Get JSON data
        data = request.get_json()

        # Extract input features
        sepal_length = data["sepal_length"]
        sepal_width = data["sepal_width"]
        petal_length = data["petal_length"]
        petal_width = data["petal_width"]

        # Create input array
        features = np.array([[
            sepal_length,
            sepal_width,
            petal_length,
            petal_width
        ]])

        # Scale the input
        features_scaled = scaler.transform(features)

        # Make prediction
        prediction = model.predict(features_scaled)[0]

        # Get prediction probability
        probability = model.predict_proba(features_scaled)[0]

        confidence = probability[prediction]

        # Return result
        return jsonify({
            "prediction": class_names[prediction],
            "confidence": round(float(confidence), 4)
        })

    except Exception as e:

        return jsonify({
            "error": str(e)
        }), 400


# --------------------------------------------------------
# RUN FLASK APPLICATION
# --------------------------------------------------------

if __name__ == "__main__":

    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False
    )

'''

# Save Flask code
with open("app.py", "w") as file:
    file.write(flask_code)

print("\nFlask API file created successfully: app.py")


# ------------------------------------------------------------
# 11. DISPLAY PROJECT FILES
# ------------------------------------------------------------

print("\n================ PROJECT FILES ================")

for file in os.listdir():

    if file in [
        "app.py",
        "iris_model.pkl",
        "scaler.pkl"
    ]:

        print("✓", file)


# ------------------------------------------------------------
# 12. TEST MODEL DIRECTLY
# ------------------------------------------------------------

print("\n================ SAMPLE PREDICTION ================")

# Example flower measurements
sample = np.array([[
    5.1,
    3.5,
    1.4,
    0.2
]])

# Scale sample
sample_scaled = scaler.transform(sample)

# Predict
prediction = model.predict(sample_scaled)[0]

# Probability
probability = model.predict_proba(sample_scaled)[0]

print("Input:")
print("Sepal Length:", 5.1)
print("Sepal Width :", 3.5)
print("Petal Length:", 1.4)
print("Petal Width :", 0.2)

print("\nPredicted Flower:", iris.target_names[prediction])

print(
    "Confidence:",
    round(float(probability[prediction]) * 100, 2),
    "%"
)


# ------------------------------------------------------------
# 13. CREATE REQUIREMENTS.TXT
# ------------------------------------------------------------

requirements = """flask
joblib
numpy
scikit-learn
requests
"""

with open("requirements.txt", "w") as file:
    file.write(requirements)

print("\nrequirements.txt created successfully!")


# ------------------------------------------------------------
# 14. CREATE README FILE
# ------------------------------------------------------------

readme = """
# Yuva Intern Week 4 - AI Project Deployment & Capstone

## Project Title
Iris Flower Prediction using Machine Learning and Flask API

## Description
This project is an end-to-end Machine Learning application that
predicts the species of an Iris flower based on its measurements.

## Technologies Used
- Python
- Scikit-learn
- NumPy
- Joblib
- Flask
- Google Colab

## Machine Learning Algorithm
Logistic Regression

## Features
- Data preprocessing
- Model training
- Model testing
- Model evaluation
- Model serialization using Joblib
- Flask REST API
- Prediction endpoint

## Input Features
- Sepal Length
- Sepal Width
- Petal Length
- Petal Width

## Output
The API predicts one of the following:
- Setosa
- Versicolor
- Virginica

## API Endpoint

POST /predict

## Example Input

{
    "sepal_length": 5.1,
    "sepal_width": 3.5,
    "petal_length": 1.4,
    "petal_width": 0.2
}

## Example Output

{
    "prediction": "setosa",
    "confidence": 0.98
}

## Author
Shubhakarini

## Internship
Yuva Intern - Week 4
"""

with open("README.md", "w") as file:
    file.write(readme)

print("README.md created successfully!")


# ------------------------------------------------------------
# 15. FINAL PROJECT SUMMARY
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("YUVA INTERN WEEK 4 PROJECT COMPLETED")
print("=" * 60)

print("\nProject: Iris Flower Prediction API")
print("Algorithm: Logistic Regression")
print("Preprocessing: StandardScaler")
print("Serialization: Joblib")
print("Deployment Framework: Flask")

print("\nFiles created:")

print("1. app.py")
print("2. iris_model.pkl")
print("3. scaler.pkl")
print("4. requirements.txt")
print("5. README.md")

print("\nProject completed successfully!")

Dataset loaded successfully!
Dataset shape: (150, 4)
Target names: ['setosa' 'versicolor' 'virginica']

Data split completed!
Training samples: 120
Testing samples: 30

Data preprocessing completed!

Model training completed!

================ MODEL EVALUATION ================
Accuracy: 0.9333333333333333

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30


Model saved as: iris_model.pkl
Scaler saved as: scaler.pkl

Saved files:
- iris_model.pkl
- scaler.pkl

Flask API file created successfully: app.py

================ PROJECT FILES ================
✓ app.py
✓ iris_model.pkl
✓ scaler.pkl

================ SAMPLE PREDICTION ===============